In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv("./payload_full.csv")

In [3]:
df.sample(5)

,payload,length,attack_type,label
18240,challee@zonamodelos.co,22,norm,norm
25181,7912641322144923,16,norm,norm
13168,"1')));select * from generate_series(7754,7754,...",96,sqli,anom
3464,lact1ad1,8,norm,norm
258,4867284155125635,16,norm,norm


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31067 entries, 0 to 31066
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   payload      31067 non-null  object
 1   length       31067 non-null  int64 
 2   attack_type  31067 non-null  object
 3   label        31067 non-null  object
dtypes: int64(1), object(3)
memory usage: 971.0+ KB


In [6]:
df["attack_type"].value_counts()

attack_type
norm              19304
sqli              10852
xss                 532
path-traversal      290
cmdi                 89
Name: count, dtype: int64

In [7]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    text = " ".join([word for word in text.split() if word not in stop_words])  # Remove stopwords
    return text

df["cleaned_payload"] = df["payload"].apply(clean_text)


[nltk_data] Downloading package stopwords to C:\Users\hp/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
df["attack_type_encoded"] = label_encoder.fit_transform(df["attack_type"])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(df["cleaned_payload"], df["attack_type_encoded"], test_size=0.2, random_state=42, stratify=df["attack_type_encoded"])

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 24853
Testing Samples: 6214


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)  # Limiting to 5000 features
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Vectorization Complete!")


TF-IDF Vectorization Complete!


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train model
log_model = LogisticRegression()
log_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred_log = log_model.predict(X_test_tfidf)

# Evaluate
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log, target_names=label_encoder.classes_))


Logistic Regression Accuracy: 0.9628258770518184
                precision    recall  f1-score   support

          cmdi       1.00      0.39      0.56        18
          norm       0.94      1.00      0.97      3861
path-traversal       0.98      0.78      0.87        58
          sqli       1.00      0.93      0.96      2171
           xss       1.00      0.44      0.61       106

      accuracy                           0.96      6214
     macro avg       0.98      0.71      0.80      6214
  weighted avg       0.96      0.96      0.96      6214



In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train_tfidf, y_train)

y_pred_rf = rf_model.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_))


Random Forest Accuracy: 0.9678146121660766
                precision    recall  f1-score   support

          cmdi       1.00      0.67      0.80        18
          norm       0.95      1.00      0.97      3861
path-traversal       0.98      0.78      0.87        58
          sqli       1.00      0.93      0.96      2171
           xss       1.00      0.69      0.82       106

      accuracy                           0.97      6214
     macro avg       0.99      0.81      0.88      6214
  weighted avg       0.97      0.97      0.97      6214



In [12]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, target_names=label_encoder.classes_))


Naive Bayes Accuracy: 0.9568715803025426
                precision    recall  f1-score   support

          cmdi       0.00      0.00      0.00        18
          norm       0.94      1.00      0.97      3861
path-traversal       1.00      0.53      0.70        58
          sqli       1.00      0.93      0.96      2171
           xss       1.00      0.29      0.45       106

      accuracy                           0.96      6214
     macro avg       0.79      0.55      0.62      6214
  weighted avg       0.96      0.96      0.95      6214



C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

In [13]:
models = {
    "Logistic Regression": accuracy_score(y_test, y_pred_log),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    "Naive Bayes": accuracy_score(y_test, y_pred_nb)
}

print(models)


{'Logistic Regression': 0.9628258770518184, 'Random Forest': 0.9678146121660766, 'Naive Bayes': 0.9568715803025426}


In [14]:
y_pred_rf = rf_model.predict(X_test_tfidf)

In [15]:
y_test_original = label_encoder.inverse_transform(y_test)
y_pred_original = label_encoder.inverse_transform(y_pred_rf)

In [17]:
comparison_sample = X_train.sample(10, random_state=42)

In [18]:
comparison_sample

2602                                      5685407082678044
15366             7731 make_set255143894389 unly like unly
8041     scheff_guiffreinterpretaciondiccionarioysignif...
23473                                                piero
9829                                             ausrbinid
7921                                                 31494
23372                                            29967631b
9428                                     repraz torrubiano
9776     1select frrk dual 31453145 extractvalue7982con...
29701                                9509 93039303 order 1
Name: cleaned_payload, dtype: object

In [20]:
import gradio as gr
import pandas as pd
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load dataset (Assuming dataset is available as 'df')
df = pd.read_csv("payload_full.csv")  # Replace with actual path

# Encode labels
label_encoder = LabelEncoder()
df["attack_type_encoded"] = label_encoder.fit_transform(df["attack_type"])

# Text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df["cleaned_payload"] = df["payload"].apply(clean_text)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(df["cleaned_payload"], df["attack_type_encoded"], test_size=0.2, random_state=42, stratify=df["attack_type_encoded"])

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

def predict_attack(payload):
    cleaned_payload = clean_text(payload)
    payload_tfidf = vectorizer.transform([cleaned_payload])
    prediction = rf_model.predict(payload_tfidf)
    return label_encoder.inverse_transform(prediction)[0]

# Gradio Interface
demo = gr.Interface(
    fn=predict_attack,
    inputs=gr.Textbox(label="Enter Payload"),
    outputs=gr.Textbox(label="Predicted Attack Type"),
    title="Attack Type Prediction",
    description="Enter a payload to predict its attack type using a trained Random Forest model."
)

demo.launch()


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [24]:
pip install pickle

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pickle


In [27]:
import pickle

In [28]:
with open("random_forest_model.pkl", "wb") as model_file:
    pickle.dump(rf_model, model_file)

with open("tfidf_vectorizer.pkl", "wb") as vectorizer_file:
    pickle.dump(vectorizer, vectorizer_file)

In [29]:
import pickle
from sklearn.preprocessing import LabelEncoder

# Assuming `label_encoder` is already fitted
with open("label_encoder.pkl", "wb") as le_file:
    pickle.dump(label_encoder.classes_, le_file)
